In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import pickle
import json

In [3]:

df = pd.read_csv(r'D:\Miini_project2\data\cleaned_data.csv')

In [5]:
# Clean column names
df.columns = [col.strip().replace(' ', '_').replace('(', '').replace(')', '') for col in df.columns]

# Use only features that exist in the dataframe
all_columns = set(df.columns)
categorical_features = [col for col in ['gender', 'category_name', 'payment_method', 'city'] if col in all_columns]
numeric_features = [col for col in ['quantity', 'price', 'age', 'price_per_item'] if col in all_columns]
target = 'price' if 'price' in all_columns else df.columns[-1]  # fallback to last column if needed

print('Categorical features:', categorical_features)
print('Numeric features:', numeric_features)
print('Target:', target)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer([
    ('num', 'passthrough', numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
], remainder='drop')

X = df[categorical_features + numeric_features]
y = df[target]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
model.fit(X_train_processed, y_train)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2': r2_score(y_test, y_pred)
    }

metrics = evaluate_model(model, X_test_processed, y_test)

artifacts = {
    'model': model,
    'preprocessor': preprocessor,
    'metrics': metrics,
    'feature_names': list(X.columns)
}

Categorical features: ['gender', 'category_name', 'payment_method', 'city']
Numeric features: ['quantity', 'price', 'age', 'price_per_item']
Target: price


In [9]:
with open(r'D:\Miini_project2\data_clean\data_cleaning.ipynb.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

In [11]:

metadata = {
    'model_type': 'RandomForestRegressor',
    'version': '1.0',
    'features': {
        'categorical': categorical_features,
        'numeric': numeric_features
    },
    'performance': metrics
}

with open(r'D:\Miini_project2\appmodel_metaatad.json', 'w') as f:
    json.dump(metadata, f, indent=2)

In [17]:
# --- Regression Models: Linear Regression, Random Forest, SVR, Ridge ---
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Use preprocessed features for regression models
X_train_reg = X_train_processed if 'X_train_processed' in locals() else X_train
X_test_reg = X_test_processed if 'X_test_processed' in locals() else X_test

import pandas as pd
def is_classification_target(y):
    return (pd.api.types.is_object_dtype(y) or isinstance(y.dtype, pd.CategoricalDtype) or y.nunique() < 20)

if not is_classification_target(y_train):
    # Linear Regression
    try:
        lr = LinearRegression()
        lr.fit(X_train_reg, y_train)
        lr_pred = lr.predict(X_test_reg)
        print("Linear Regression:")
        print("  MAE:", mean_absolute_error(y_test, lr_pred))
        print("  RMSE:", np.sqrt(mean_squared_error(y_test, lr_pred)))
        print("  R2:", r2_score(y_test, lr_pred))
    except Exception as e:
        print("Linear Regression failed:", e)

    # Random Forest Regressor
    try:
        rf = RandomForestRegressor(n_estimators=100, random_state=42)
        rf.fit(X_train_reg, y_train)
        rf_pred = rf.predict(X_test_reg)
        print("Random Forest Regressor:")
        print("  MAE:", mean_absolute_error(y_test, rf_pred))
        print("  RMSE:", np.sqrt(mean_squared_error(y_test, rf_pred)))
        print("  R2:", r2_score(y_test, rf_pred))
    except Exception as e:
        print("Random Forest Regressor failed:", e)

    # Support Vector Regressor
    try:
        svr = SVR()
        svr.fit(X_train_reg, y_train)
        svr_pred = svr.predict(X_test_reg)
        print("Support Vector Regressor:")
        print("  MAE:", mean_absolute_error(y_test, svr_pred))
        print("  RMSE:", np.sqrt(mean_squared_error(y_test, svr_pred)))
        print("  R2:", r2_score(y_test, svr_pred))
    except Exception as e:
        print("Support Vector Regressor failed:", e)

    # Ridge Regression
    try:
        ridge = Ridge()
        ridge.fit(X_train_reg, y_train)
        ridge_pred = ridge.predict(X_test_reg)
        print("Ridge Regression:")
        print("  MAE:", mean_absolute_error(y_test, ridge_pred))
        print("  RMSE:", np.sqrt(mean_squared_error(y_test, ridge_pred)))
        print("  R2:", r2_score(y_test, ridge_pred))
    except Exception as e:
        print("Ridge Regression failed:", e)
else:
    print("Target variable is suitable for classification. Please use classification models for this target.")

Linear Regression:
  MAE: 7.149171284240442e-05
  RMSE: 8.971680975184578e-05
  R2: 0.999999999999564
Random Forest Regressor:
  MAE: 0.42665999999998544
  RMSE: 0.5652044139833018
  R2: 0.9999826985913052
Support Vector Regressor:
  MAE: 39.0341727838763
  RMSE: 50.13727474003489
  R2: 0.8638580950709082
Ridge Regression:
  MAE: 0.021185137941890128
  RMSE: 0.024563946965150674
  R2: 0.9999999673211566
Random Forest Regressor:
  MAE: 0.42665999999998544
  RMSE: 0.5652044139833018
  R2: 0.9999826985913052
Support Vector Regressor:
  MAE: 39.0341727838763
  RMSE: 50.13727474003489
  R2: 0.8638580950709082
Ridge Regression:
  MAE: 0.021185137941890128
  RMSE: 0.024563946965150674
  R2: 0.9999999673211566
